# Tutorial 1.11: LangGraph Deep Agents with MLflow Tracing

**Deep Agents** are LangChain's open-source agent harness for long-running, multi-step autonomous tasks. Built on LangGraph, they add built-in **planning** (todo lists), **file system tools** (read/write/edit files), and **sub-agent delegation** on top of the standard tool-calling loop.

### What Makes Deep Agents Different from Regular LangGraph Agents?

| Feature | Standard LangGraph Agent | Deep Agent |
|---------|------------------------|------------|
| Planning | Manual | Built-in `write_todos` / `read_todos` |
| Context management | Limited by context window | File system tools offload to storage |
| Delegation | Single agent | Sub-agents via `task()` tool |
| Memory | Per-conversation | Persistent across threads |

### What You'll Learn

1. **Example 1** — Basic Deep Agent with tools and planning
2. **Example 2** — File system context management
3. **Example 3** — Sub-agent delegation (multi-agent)
4. **Example 4** — Evaluating Deep Agent outputs with `mlflow.genai.evaluate()`

All examples use `mlflow.langchain.autolog()` to automatically trace the full agent execution graph.

### Prerequisites
- Completed tutorials 01-07 (MLflow basics, tracing, framework integrations)
- OpenAI API key configured in `.env`

---
## Step 1: Environment Setup

In [1]:
# Install deepagents (if not already installed via pyproject.toml)
%pip install -q deepagents

/Users/jules/git-repos/mlflow-genai-tutorials/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import mlflow
from dotenv import load_dotenv

load_dotenv()

# Configure MLflow
mlflow.set_tracking_uri(os.environ.get("MLFLOW_TRACKING_URI", "http://localhost:5000"))
mlflow.set_experiment("11-deep-agents-langgraph")

# Enable auto-tracing for LangChain/LangGraph (covers Deep Agents)
mlflow.langchain.autolog()

In [3]:
from deepagents import create_deep_agent
from langchain.chat_models import init_chat_model

# Initialize the LLM - uses OpenAI by default
llm = init_chat_model("openai:gpt-5-mini")
print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")
print(f"Experiment: {mlflow.get_experiment_by_name('11-deep-agents-langgraph').name}")

MLflow tracking URI: http://localhost:5000
Experiment: 11-deep-agents-langgraph


---
## Example 1: Basic Deep Agent — Research & Summarize

This example creates a Deep Agent with custom tools that researches a topic and produces a structured summary. The agent automatically uses its built-in **planning tools** (`write_todos`, `read_todos`) to decompose the task into steps.

![Example 1: Basic Deep Agent](images/11_deep_agents_ex1_basic.svg)

**What to observe in MLflow traces:**
- Planning spans showing the agent breaking down the task
- Individual tool call spans for each research step
- LLM reasoning spans between actions

In [4]:
# Define custom tools the agent can use to search a knowledge base.
# This is a simple knowledge base that we'll use to test the agent.
# In a real use case, this would be a much larger and more complex knowledge base, most
# likely stored in a database or a vector database.
# For this example, we'll just use a small knowledge base of 4 topics.

def search_knowledge_base(query: str) -> str:
    """Search an internal knowledge base for information about MLflow, GenAI, and Agentic Workflow topics.
    Returns relevant information snippets."""

    knowledge = {
        "mlflow tracing": (
            "MLflow Tracing provides observability for AI applications. It captures "
            "hierarchical spans showing LLM calls, tool invocations, and retrieval steps. "
            "Supports auto-tracing for OpenAI, LangChain, LlamaIndex, and LangGraph, and many more. "
            "Traces are stored in the MLflow tracking server and viewable in the UI. With Managved MLflow traces configured to be stores in Unity Catalog, you can also access them in Databricks SQL."
        ),
        "mlflow evaluation": (
            "MLflow GenAI Evaluation (mlflow.genai.evaluate) assesses AI application quality "
            "using built-in scorers (Correctness, RelevanceToQuery, Safety) and custom scorers. "
            "It integrates with tracing to evaluate end-to-end agent behavior. "
            "Supports LLM-as-a-judge and Agent-as-a-judge patterns for automated quality assessment."
        ),
        "deep agents": (
            "Deep Agents are LangChain's open-source agent harness built on LangGraph. "
            "They add planning (todo lists), file system tools, and sub-agent delegation "
            "to the standard tool-calling loop. Designed for long-running, multi-step tasks "
            "like research, coding, and analysis."
        ),
        "langgraph": (
            "LangGraph is a framework for building stateful, multi-actor AI applications "
            "using graph-based workflows. It supports conditional routing, cycles, "
            "checkpointing, and streaming. Agents built with LangGraph are automatically "
            "traced by MLflow when mlflow.langchain.autolog() is enabled."
        ),
    }
    # Simple keyword matching
    results = []
    for topic, info in knowledge.items():
        if any(word in query.lower() for word in topic.split()):
            results.append(f"[{topic.upper()}]: {info}")
    return "\n\n".join(results) if results else f"No results found for: {query}"


def get_latest_stats(category: str) -> str:
    """Get the latest statistics for a given MLflow/AI category."""
    stats = {
        "adoption": "MLflow has 20M+ monthly downloads, 100K+ GitHub stars, and 1000+ contributors.",
        "performance": "GPT-4o averages 250ms first-token latency. Claude Sonnet: 200ms.",
        "cost": "GPT-4o: $2.50/1M input tokens. Claude Sonnet: $3/1M input tokens.",
    }
    return stats.get(category.lower(), f"No stats available for: {category}")


print("Tools defined: search_knowledge_base, get_latest_stats")

Tools defined: search_knowledge_base, get_latest_stats


In [5]:
research_agent = create_deep_agent(
    model=llm,
    tools=[search_knowledge_base, get_latest_stats],
    system_prompt=(
        "You are a research assistant. When given a research topic:\n"
        "1. Use write_todos to plan your research steps\n"
        "2. Search the knowledge base for relevant information\n"
        "3. Gather supporting statistics\n"
        "4. Synthesize findings into a structured summary with sections: "
        "Overview, Key Findings, Statistics, and Conclusion"
    ),
)

print(f"Deep Agent created — type: {type(research_agent)}")

Deep Agent created — type: <class 'langgraph.graph.state.CompiledStateGraph'>


In [6]:
result = research_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": (
                "Research how MLflow provides observability for AI agents. "
                "Cover tracing capabilities, evaluation features, and "
                "how it integrates with frameworks like LangGraph and Deep Agents."
            ),
        }
    ]
})

print(result["messages"][-1].content)

Overview
- MLflow provides end-to-end observability for agentic and generative AI applications via two complementary capabilities:
  1. Tracing: hierarchical, span-based traces that capture LLM calls, tool invocations, retrievals, and sub-agent activity.
  2. Evaluation: GenAI-focused evaluation primitives (mlflow.genai.evaluate) that compute built-in and custom scorers and can operate over traces to assess end-to-end agent behavior.
- Traces and evaluation outputs are persisted to the MLflow tracking server (viewable in the MLflow UI) and — when using Managed MLflow with Unity Catalog — accessible via Databricks SQL.

Key findings (tracing, evaluation, integrations)

1) Tracing capabilities
- Hierarchical spans: MLflow Tracing records a hierarchical span model that shows the call stack of an agent run (top-level agent, planner/manager, LLM calls, tool or retrieval calls, sub-agent runs).
- What it captures: prompts/responses, LLM API calls, tool calls, retrieval hits, intermediate rea

Trace(trace_id=tr-85161a7a37cc25056494f7a91056e2d7)

### What to Look for in the MLflow UI

Open the MLflow UI at `http://localhost:5000` and navigate to the **11-deep-agents-langgraph** experiment.

In the trace view you should see:
- **Root span**: The full `invoke` call on the compiled LangGraph graph
- **Planning spans**: `write_todos` calls showing how the agent decomposed the task
- **Tool call spans**: `search_knowledge_base` and `get_latest_stats` invocations with inputs/outputs
- **LLM spans**: Each `gpt-5-mini` call with the full prompt and response
- **Hierarchical nesting**: Planning → tool calls → synthesis, all nested under the root

---
## Example 2: File System Context Management

Deep Agents manage context by reading and writing files rather than keeping everything in the LLM's context window. This example demonstrates an agent that reads a draft document, identifies improvements, and applies edits — all using the built-in file system tools.

![Example 2: File System Context Management](images/11_deep_agents_ex2_filesystem.svg)

We use a **`FilesystemBackend`** with a local workspace directory. The agent gets access to `read_file`, `write_file`, `edit_file`, `ls`, and `glob` automatically.

**What to observe in MLflow traces:**
- File operation spans (`read_file`, `write_file`, `edit_file`)
- LLM reasoning spans between each file operation
- How the agent offloads intermediate work to the file system

In [7]:
import shutil
import os
from deepagents.backends import FilesystemBackend

# Create a workspace directory with an initial draft document for the agent to work on
# and expand on it in the next steps.
workspace_dir = "./agent_workspace"
os.makedirs(workspace_dir, exist_ok=True)

draft_content = """# MLflow GenAI Platform Overview

MLflow is a tool for machine learning. It does tracking and stuff.

## Tracing
MLflow can trace things. It works with some frameworks.

## Evaluation
You can evaluate models with MLflow. It has some scorers.

## Conclusion
MLflow is good for ML.
"""

with open(os.path.join(workspace_dir, "draft.md"), "w") as f:
    f.write(draft_content)

print(f"Workspace created at: {workspace_dir}")
print(f"Draft document written ({len(draft_content)} chars)")

Workspace created at: ./agent_workspace
Draft document written (280 chars)


In [8]:
editor_agent = create_deep_agent(
    model=llm,
    system_prompt=(
        "You are a technical editor. Your task:\n"
        "1. Read the draft document at /draft.md\n"
        "2. Identify 3 specific improvements (vague language, missing details, weak structure)\n"
        "3. Write your improvement plan to /edit_plan.md\n"
        "4. Apply each improvement by editing the draft\n"
        "5. Write the final polished version to /final.md\n\n"
        "Be specific and substantive in your edits. Replace vague statements with "
        "concrete technical details about MLflow's GenAI capabilities."
    ),
    backend=FilesystemBackend(
        root_dir=workspace_dir,
        virtual_mode=True,  # Restricts file access to the workspace
    ),
)

print("Editor agent created with FilesystemBackend")

Editor agent created with FilesystemBackend


In [9]:
edit_result = editor_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Please review and improve the draft at /draft.md. It needs to be more specific and technically accurate.",
        }
    ]
})

print(edit_result["messages"][-1].content)

Identified 3 specific improvements and what I changed

1) Replace vague language with concrete MLflow components and APIs
- Replaced "tracking and stuff" and similar vague lines with precise descriptions of MLflow primitives (Tracking, Models/flavors, Model Registry, Serving) and explicit API calls such as mlflow.start_run, mlflow.log_param, mlflow.log_metric, mlflow.log_artifact, mlflow.pytorch.log_model, mlflow.pyfunc.log_model, mlflow.models.signature.infer_signature, and MlflowClient.create_model_version / transition_model_version_stage.

2) Add GenAI-specific technical details
- Added concrete patterns for logging prompts and generations (JSONL format example), saving tokenizers and vocab as artifacts, storing embeddings (numpy .npy + metadata), capturing model signatures and input_examples, and recommended evaluation metrics (perplexity, BLEU/ROUGE, BERTScore, FID for images). Included examples showing how to log these items and metrics.

3) Improve structure and provide an actio

Trace(trace_id=tr-e64e15b88e260eb2ebf7a9f22579cc3a)

In [10]:
for filename in ["edit_plan.md", "final.md"]:
    filepath = os.path.join(workspace_dir, filename)
    if os.path.exists(filepath):
        with open(filepath) as f:
            content = f.read()
        print(f"\n{'='*60}")
        print(f"📄 {filename}")
        print(f"{'='*60}")
        print(content[:500])
        if len(content) > 500:
            print(f"... ({len(content)} chars total)")
    else:
        print(f"⚠️  {filename} not found — agent may have used different file names")


📄 edit_plan.md
Improvement plan for /draft.md

1) Replace vague high-level overview with a precise technical summary of MLflow and how its components (Tracking, Models, Model Registry, Projects, Serving) support Generative AI workflows. Provide concrete component responsibilities and common integrations (PyTorch/Transformers, tokenizers, Hugging Face artifacts).

2) Expand "Tracing" into a detailed "Tracking" section that shows concrete APIs and patterns for logging GenAI experiments: sample mlflow.start_run()
... (1906 chars total)

📄 final.md
MLflow GenAI Platform Overview

MLflow is an open-source platform to manage the ML lifecycle. For Generative AI workflows, MLflow provides concrete primitives for experiment tracking, model packaging, model versioning (Model Registry), reproducible projects, and serving. This document describes how to apply MLflow to GenAI use cases (large language models, image/video generative models) with concrete API patterns and best practices.

Tracking
U

---
## Example 3: Sub-Agent Delegation (Multi-Agent)

Deep Agents can spawn **sub-agents** to handle specialized subtasks. The parent agent coordinates work by delegating via the built-in `task()` tool, and each sub-agent runs in its own context — keeping the parent's context clean.

![Example 3: Sub-Agent Delegation](images/11_deep_agents_ex3_subagents.svg)

This example creates a **Technical Report Coordinator** that delegates to three specialists:
- **Researcher** — gathers information on a topic
- **Analyst** — synthesizes findings and identifies trends
- **Writer** — produces the final report

**What to observe in MLflow traces:**
- Parent agent span containing child sub-agent spans
- Each sub-agent has its own planning and tool-calling hierarchy
- Clear delegation boundaries visible in the trace tree

In [11]:
coordinator_agent = create_deep_agent(
    model=llm,
    system_prompt=(
        "You are a Technical Report Coordinator. When asked to produce a report:\n"
        "1. Delegate research to the 'researcher' sub-agent\n"
        "2. Send the research findings to the 'analyst' sub-agent for analysis\n"
        "3. Send the analysis to the 'writer' sub-agent to produce the final report\n"
        "4. Review the final report and present it to the user\n\n"
        "Use the task() tool to delegate work to each sub-agent. "
        "Provide clear, specific instructions to each sub-agent."
    ),
    tools=[search_knowledge_base, get_latest_stats],
    subagents=[
        {
            "name": "researcher",
            "description": "Gathers information from the knowledge base and collects statistics",
            "system_prompt": (
                "You are a research specialist. Use the available tools to gather "
                "comprehensive information on the assigned topic. Return your findings "
                "as a structured list of key facts and data points."
            ),
            "tools": [search_knowledge_base, get_latest_stats],
        },
        {
            "name": "analyst",
            "description": "Analyzes research findings and identifies key trends and insights",
            "system_prompt": (
                "You are a data analyst. Given research findings, identify the top 3 trends, "
                "key insights, and any gaps in the data. Structure your analysis with: "
                "Trends, Insights, and Recommendations sections."
            ),
        },
        {
            "name": "writer",
            "description": "Produces polished technical reports from analysis",
            "system_prompt": (
                "You are a technical writer. Given an analysis, produce a concise, "
                "well-structured report with: Executive Summary, Detailed Findings, "
                "and Actionable Recommendations. Use clear, professional language."
            ),
        },
    ],
)

print("Coordinator agent created with 3 sub-agents: researcher, analyst, writer")

Coordinator agent created with 3 sub-agents: researcher, analyst, writer


In [12]:
coordinator_result = coordinator_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": (
                "Produce a technical report on the state of AI observability. "
                "Cover how MLflow tracing and evaluation help teams monitor and "
                "improve their AI agents. Include adoption statistics."
            ),
        }
    ]
})

print(coordinator_result["messages"][-1].content)

Report: "Observability for Agentic GenAI Systems: Metrics, Gaps, and Implementation Guidance"

Executive summary
Agentic GenAI systems (multi-step orchestrations of LLM calls, retrieval, tool executions, and external APIs) require observability that spans model quality, system reliability, safety, and cost. MLflow remains an essential platform for experiment tracking, model registry, and offline evaluation; recent MLflow GenAI features (tracing and mlflow.genai.evaluate) improve linkage between runs, traces, and automated evaluation. However, MLflow does not replace runtime distributed tracing, high-cardinality analytics, real-time alerting, PII/redaction pipelines, or long-term searchable trace stores. Practical observability combines MLflow for lineage/experiments with OpenTelemetry/Jaeger/Honeycomb for runtime traces, Prometheus/Grafana or APMs for metrics, and specialized monitoring tools (Arize, WhyLabs, Evidently, Fiddler) for drift, explainability and safety. This report present

Trace(trace_id=tr-0738fc3ba4fb9d5de9f267465302a237)

### Trace Comparison: Single Agent vs. Multi-Agent

Compare the traces from Example 1 and Example 3 in the MLflow UI:

| Aspect | Example 1 (Single Agent) | Example 3 (Coordinator + Sub-Agents) |
|--------|-------------------------|--------------------------------------|
| Depth | 2-3 levels (agent → tools) | 4-5 levels (coordinator → sub-agent → tools) |
| Breadth | Sequential tool calls | Parallel-looking delegation spans |
| Context | All in one agent | Each sub-agent has isolated context |
| Visibility | Flat tool calls | Clear responsibility boundaries |

The hierarchical trace structure in Example 3 makes it easy to debug which sub-agent produced which part of the output — a key advantage for production multi-agent systems.

---
## Example 4: Evaluating Deep Agent Outputs with MLflow

Building and tracing agents is only half the story — you also need to **evaluate** their outputs systematically. This example uses `mlflow.genai.evaluate()` to assess the research agent from Example 1 across multiple queries.

We'll use:
- **`RelevanceToQuery`** — Is the response relevant to what was asked?
- **`Safety`** — Does the response contain harmful content?
- **`Guidelines`** — Custom rubric for research completeness

In [13]:
import pandas as pd
from mlflow.genai.scorers import RelevanceToQuery, Safety, Guidelines

eval_data = pd.DataFrame({
    "inputs": [
        {"query": "What is MLflow tracing and how does it work?"},
        {"query": "How do Deep Agents compare to standard LangGraph agents?"},
        {"query": "What evaluation capabilities does MLflow provide for GenAI?"},
        {"query": "How does LangGraph enable stateful agent workflows?"},
    ],
})

print(f"Evaluation dataset: {len(eval_data)} queries")
eval_data

Evaluation dataset: 4 queries


,inputs
0,{'query': 'What is MLflow tracing and how does...
1,{'query': 'How do Deep Agents compare to stand...
2,{'query': 'What evaluation capabilities does M...
3,{'query': 'How does LangGraph enable stateful ...


In [14]:
def research_predict(query: str) -> str:
    """Run the research agent and return its response."""
    result = research_agent.invoke({
        "messages": [{"role": "user", "content": query}]
    })
    return result["messages"][-1].content

research_quality_guidelines = Guidelines(
    name="research_completeness",
    guidelines=(
        "The response should be a well-structured research summary that includes: "
        "(1) An overview or introduction to the topic, "
        "(2) Specific technical details and facts (not vague generalizations), "
        "(3) Multiple aspects or dimensions of the topic covered, "
        "(4) A clear conclusion or synthesis. "
        "Responses that are too brief, overly vague, or miss key aspects should score lower."
    ),
)

print("Predict function and custom scorer defined")

Predict function and custom scorer defined


In [15]:
eval_results = mlflow.genai.evaluate(
    data=eval_data,
    predict_fn=research_predict,
    scorers=[
        RelevanceToQuery(),
        Safety(),
        research_quality_guidelines,
    ],
)

print("Evaluation complete!")
eval_results.metrics

2026/04/10 11:43:16 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.
2026/04/10 11:43:16 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.
2026/04/10 11:43:16 WARNING mlflow.tracing.fluent: Failed to start span LangGraph: 'NonRecordingSpan' object has no attribute 'context'. For full traceback, set logging level to debug.


Evaluating:   0%|          | 0/4 [Elapsed: 00:00, Remaining: ?] 

Evaluation complete!


{'safety/mean': np.float64(1.0),
 'relevance_to_query/mean': np.float64(1.0),
 'research_completeness/mean': np.float64(0.75)}

In [16]:
eval_results.tables["eval_results"]

,trace_id,safety/value,relevance_to_query/value,research_completeness/value,trace,client_request_id,state,request_time,execution_duration,request,response,trace_metadata,tags,spans,assessments
0,tr-58080d562fe8d0bd3a154c719742222c,yes,yes,no,"{""info"": {""trace_id"": ""tr-58080d562fe8d0bd3a15...",None,OK,1775846640848,14694,"{'messages': [{'role': 'user', 'content': 'Wha...",{'messages': [{'content': 'What is MLflow trac...,"{'mlflow.traceOutputs': '{""messages"": [{""conte...",{'mlflow.eval.requestId': '0e4609ad-478f-43ee-...,"[{'trace_id': 'WAgNVi/o0L06FUxxl0IiLA==', 'spa...",[{'assessment_id': 'a-8a2aaf342a224db8b67dd4ad...
1,tr-28ad191a8af2724cbb603b2b9a08be43,yes,yes,yes,"{""info"": {""trace_id"": ""tr-28ad191a8af2724cbb60...",None,OK,1775846640851,47531,"{'messages': [{'role': 'user', 'content': 'How...",{'messages': [{'content': 'How do Deep Agents ...,"{'mlflow.traceOutputs': '{""messages"": [{""conte...",{'mlflow.eval.requestId': '9e532d00-966e-4dfb-...,"[{'trace_id': 'KK0ZGorycky7YDsrmgi+Qw==', 'spa...",[{'assessment_id': 'a-eb19dacdfcde46d2be2b585e...
2,tr-5274c9de00f619bf22f4c7573469494e,yes,yes,yes,"{""info"": {""trace_id"": ""tr-5274c9de00f619bf22f4...",None,OK,1775846640852,47822,"{'messages': [{'role': 'user', 'content': 'Wha...",{'messages': [{'content': 'What evaluation cap...,"{'mlflow.traceOutputs': '{""messages"": [{""conte...",{'mlflow.eval.requestId': 'c1e13a4a-857c-4ace-...,"[{'trace_id': 'UnTJ3gD2Gb8i9MdXNGlJTg==', 'spa...",[{'assessment_id': 'a-afac0549b7424c76b8e7de25...
3,tr-d0b471a55ca81a6539b0a80da1b83a4d,yes,yes,yes,"{""info"": {""trace_id"": ""tr-d0b471a55ca81a6539b0...",None,OK,1775846640854,53055,"{'messages': [{'role': 'user', 'content': 'How...",{'messages': [{'content': 'How does LangGraph ...,"{'mlflow.traceOutputs': '{""messages"": [{""conte...",{'mlflow.eval.requestId': 'c8ef53cb-8ba3-494f-...,"[{'trace_id': '0LRxpVyoGmU5sKgNobg6TQ==', 'spa...",[{'assessment_id': 'a-10a0508baa684c5b953d139c...


---
## Best Practices & Key Takeaways

### Deep Agent Design
- **Start simple**: Use `create_deep_agent()` with just tools before adding sub-agents or file backends
- **Use planning prompts**: Encourage the agent to use `write_todos` in your system prompt for complex tasks
- **Scope sub-agents tightly**: Each sub-agent should have a clear, narrow responsibility and limited tools
- **Use `FilesystemBackend` for long tasks**: Offload intermediate state to files to avoid context window limits
- **Set `virtual_mode=True`**: Restricts file access to the workspace directory for safety

### MLflow Integration
- **`mlflow.langchain.autolog()`** traces Deep Agents automatically — no manual instrumentation needed
- **Check trace depth**: Deep Agent traces are deeper than simple chains; use the MLflow UI tree view to navigate
- **Compare agent architectures**: Use MLflow experiments to A/B test single-agent vs. multi-agent approaches
- **Evaluate systematically**: Use `mlflow.genai.evaluate()` with custom `Guidelines` scorers tailored to your agent's task

### When to Use Deep Agents vs. Standard LangGraph
| Use Deep Agents when... | Use standard LangGraph when... |
|------------------------|-------------------------------|
| Task requires multiple steps over minutes | Task completes in a single LLM call |
| Agent needs to read/write intermediate files | No file I/O needed |
| Work should be delegated to specialists | Single agent with tools is sufficient |
| Planning decomposition improves quality | Simple routing or classification |

In [17]:
shutil.rmtree(workspace_dir, ignore_errors=True)
print("Workspace cleaned up.")

Workspace cleaned up.
